# 6.6 · 高斯混合模型 / Gaussian Mixture Models (GMM) & EM

> **课程定位 / Where this fits**
> 第 6 课，**Part 6 · 无监督学习**。
> Lesson 6, **Part 6 · Unsupervised Learning**.
>
> K-Means(6.1)把每个点**硬性**塞进一个簇，而且只会画圆形的簇。
> K-Means (6.1) assigns each point **hard**ly to exactly one cluster, and only finds round (spherical) clusters.
>
> GMM 解决这两件事：它给出**软分配**（一个点"70% 属于簇 A、30% 属于簇 B"），并能拟合**椭圆形**的簇。它靠 **EM 算法**训练——EM 是机器学习里最重要的算法之一，后面在缺失数据、隐马尔可夫、主题模型里反复出现。
> GMM fixes both: it gives **soft assignments** (a point can be "70% cluster A, 30% cluster B") and fits **elliptical** clusters. It is trained with the **EM algorithm** — one of the most important algorithms in ML, reused later for missing data, HMMs, and topic models.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{x}_i$ —— 第 $i$ 个样本（$d$ 维向量）/ the $i$-th sample (a $d$-vector)
> - $K$ —— 簇 / 高斯成分的个数 / number of clusters (Gaussian components)
> - $\pi_k$ —— 第 $k$ 个成分的**混合权重**（先验概率，$\sum_k\pi_k=1$）/ mixing weight (prior) of component $k$
> - $\boldsymbol\mu_k,\ \boldsymbol\Sigma_k$ —— 第 $k$ 个高斯的均值向量与协方差矩阵 / mean and covariance of Gaussian $k$
> - $\gamma_{ik}$ —— **责任** = 点 $i$ 属于成分 $k$ 的后验概率 / **responsibility** = posterior prob. that point $i$ belongs to component $k$
> - $\mathcal{N}(\mathbf{x};\boldsymbol\mu,\boldsymbol\Sigma)$ —— 多元高斯的概率密度 / multivariate Gaussian density

> 💡 **面试相关 / Interview-relevant**
> - 解释 **EM 的 E 步和 M 步**分别在做什么（出镜率 ★★★★★）
> - GMM 与 **K-Means 的关系**（K-Means 是 GMM 的硬分配 / 球形极限）（★★★★★）
> - EM 为什么**单调收敛**（★★★★）
> - **软分配 vs 硬分配**、`covariance_type`、用 **BIC** 选簇数（★★★★）
>
> Whiteboard hits: what E/M steps do, GMM↔K-Means link, why EM converges, soft vs hard assignment.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 用"数据是从若干个高斯里抽出来的"这一**生成故事**解释 GMM。
   Explain GMM through its **generative story**: data is drawn from a mixture of Gaussians.
2. 说清 EM 要解决的**鸡生蛋问题**，以及 E 步、M 步各自在干什么。
   Articulate the **chicken-and-egg problem** EM solves, and what the E and M steps each do.
3. **从零**实现一遍 GMM 的 EM，并和 sklearn 对照。
   Implement GMM's EM **from scratch** and check it against sklearn.
4. 解释为什么 **K-Means 是 GMM 的一个特例**。
   Explain why **K-Means is a special case of GMM**.
5. 用 **BIC** 选成分数，并理解 `covariance_type`。
   Choose the number of components with **BIC** and understand `covariance_type`.

## 目录 / TOC
1. [先建直觉：软分配与"混合"](#1)
2. [生成模型：数据是怎么来的 ⭐](#2)
3. [鸡生蛋问题与 EM ⭐](#3)
4. [🌋 数据：Old Faithful 间歇泉](#4)
5. [从零实现 EM ⭐](#5)
6. [软分配 + 椭圆簇](#6)
7. [GMM 与 K-Means 的关系 ⭐](#7)
8. [用 BIC 选簇数 + covariance_type](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 先建直觉：软分配与"混合" / Intuition First: Soft Assignment & "Mixtures"

先不谈公式，看一张图。假设有一批数据明显聚成两团（下一节的间歇泉数据就是这样）。
Let's start with a picture, no formulas. Suppose data clearly forms two blobs (the geyser data in the next section looks exactly like this).

**K-Means 的做法（硬分配）**：在两团中间画一条界线，线左边全归簇 A，线右边全归簇 B。处在交界处的点也被强行二选一。
**What K-Means does (hard assignment):** draw a boundary between the two blobs; everything left is cluster A, everything right is cluster B. Even a point sitting right on the border is forced to pick one side.

**GMM 的做法（软分配）**：不画硬界线，而是说"这个点有 80% 的把握属于 A、20% 属于 B"。交界处的点会得到接近 50/50 的概率——它**诚实地表达了不确定性**。
**What GMM does (soft assignment):** instead of a hard line, it says "this point is 80% A, 20% B". Border points get near 50/50 — GMM **honestly expresses uncertainty**.

而且 K-Means 只会画**圆形**的簇（因为它只用"到中心的距离"），GMM 用一个完整的**协方差矩阵**描述每个簇，所以能画**拉长的、有方向的椭圆**。
Also, K-Means only makes **round** clusters (it uses just "distance to center"), while GMM describes each cluster with a full **covariance matrix**, so it can make **stretched, tilted ellipses**.

> 一句话：**GMM = 软分配 + 椭圆形的 K-Means**。剩下的内容就是把这句话用概率语言精确化。
> In one line: **GMM = K-Means with soft assignment and ellipses**. The rest of this notebook makes that precise with probability.


<a id="2"></a>
## 2. 生成模型：数据是怎么来的 ⭐ / The Generative Story

理解 GMM 最好的方式，是想象数据是**怎么被"生成"出来的**。
The best way to understand GMM is to imagine how the data was **generated**.

对每一个样本 $\mathbf{x}_i$，大自然分两步产生它：
For each sample $\mathbf{x}_i$, nature produces it in two steps:

1. **先掷骰子选一个成分**：以概率 $\pi_k$ 选中第 $k$ 个高斯（$\pi_k$ 是这个成分的"占比"，所有 $\pi_k$ 加起来为 1）。
   **Roll a die to pick a component:** choose Gaussian $k$ with probability $\pi_k$ (its "share"; all $\pi_k$ sum to 1).
2. **再从那个高斯里抽一个点**：从 $\mathcal{N}(\boldsymbol\mu_k,\boldsymbol\Sigma_k)$ 采样得到 $\mathbf{x}_i$。
   **Draw a point from that Gaussian:** sample $\mathbf{x}_i$ from $\mathcal{N}(\boldsymbol\mu_k,\boldsymbol\Sigma_k)$.

如果把"选了哪个成分"这件事忘掉（边缘化掉），单看 $\mathbf{x}$ 的概率密度就是 $K$ 个高斯的**加权和**：
If we forget which component was chosen (marginalize it out), the density of $\mathbf{x}$ is a **weighted sum** of $K$ Gaussians:

$$p(\mathbf{x}) = \sum_{k=1}^{K}\pi_k\,\mathcal{N}(\mathbf{x};\boldsymbol\mu_k,\boldsymbol\Sigma_k)$$

这就是"**混合**(mixture)"二字的来历——总分布是若干高斯按权重混在一起。
This is where the word "**mixture**" comes from — the total distribution is several Gaussians blended by weight.

**我们的任务**：只看到了一堆点 $\mathbf{x}_i$，要把生成它们的参数 $\{\pi_k,\boldsymbol\mu_k,\boldsymbol\Sigma_k\}$ 估出来。
**Our task:** we only see the points $\mathbf{x}_i$, and must estimate the parameters $\{\pi_k,\boldsymbol\mu_k,\boldsymbol\Sigma_k\}$ that generated them.

**难点**：我们**不知道每个点是从哪个成分来的**。那个"选了哪个成分"的标签是看不见的——这叫**隐变量(latent variable)**。如果它可见，问题就退化成"按标签分组、各自拟合一个高斯"，非常简单。正因为它不可见，才需要 EM。
**The catch:** we **don't know which component each point came from**. That "which component" label is hidden — a **latent variable**. If it were visible, the problem would trivially reduce to "group by label, fit one Gaussian per group". Because it's hidden, we need EM.


<a id="3"></a>
## 3. 鸡生蛋问题与 EM ⭐ / The Chicken-and-Egg Problem & EM

我们陷入了一个**循环依赖**：
We're stuck in a **circular dependency**:

- **如果**知道每个点属于哪个成分 → 就能算每个成分的均值/协方差（把属于它的点拿来求统计量）。
  **If** we knew each point's component → we could compute each component's mean/covariance (just use the points belonging to it).
- **如果**知道各成分的参数 → 就能算每个点属于各成分的概率（哪个高斯更可能生成它）。
  **If** we knew the components' parameters → we could compute each point's membership probabilities (which Gaussian most likely produced it).

先有鸡还是先有蛋？**EM（期望最大化）算法**的答案是：**先随便猜一组参数，然后两步交替，越猜越准**。
Chicken or egg? The **EM (Expectation–Maximization) algorithm** says: **start with a rough guess, then alternate two steps, improving each round.**

**E 步（Expectation，期望）—— 软分配**：固定当前参数，问每个点"你有多大概率来自成分 $k$？"。这个概率就叫**责任 $\gamma_{ik}$**（成分 $k$ 对点 $i$ "负多大责任"）。它就是一个贝叶斯后验（2.8 的贝叶斯定理）：
**E-step (Expectation) — soft assignment:** fix current parameters and ask each point "how likely are you from component $k$?". This probability is the **responsibility $\gamma_{ik}$** (how much component $k$ is "responsible" for point $i$). It is just a Bayes posterior (Bayes' theorem, 2.8):

$$\gamma_{ik} = \Pr(\text{成分}=k \mid \mathbf{x}_i) = \frac{\overbrace{\pi_k}^{\text{先验}}\;\overbrace{\mathcal{N}(\mathbf{x}_i;\boldsymbol\mu_k,\boldsymbol\Sigma_k)}^{\text{似然}}}{\sum_{j=1}^{K}\pi_j\,\mathcal{N}(\mathbf{x}_i;\boldsymbol\mu_j,\boldsymbol\Sigma_j)}$$

直觉：分子是"成分 $k$ 生成这个点的可能性"（占比 × 高斯密度），分母把所有成分的可能性加起来做归一化，于是 $\sum_k\gamma_{ik}=1$。这就是那个"80% A、20% B"。
Intuition: the numerator is "how plausibly component $k$ produced this point" (share × Gaussian density); the denominator normalizes over all components so $\sum_k\gamma_{ik}=1$. This is the "80% A, 20% B".

**M 步（Maximization，最大化）—— 重估参数**：现在每个点对每个成分有了一份"软归属" $\gamma_{ik}$，就用它当**权重**重新计算参数。一个点越"负责"于成分 $k$，它对成分 $k$ 的均值/协方差贡献越大：
**M-step (Maximization) — re-estimate parameters:** now each point has a soft membership $\gamma_{ik}$; use it as a **weight** to recompute parameters. The more a point is "responsible to" component $k$, the more it contributes to component $k$'s mean/covariance:

$$N_k=\sum_{i}\gamma_{ik}\ \ (\text{成分}k\text{的"软计数"}),\qquad
\boldsymbol\mu_k=\frac{1}{N_k}\sum_i\gamma_{ik}\,\mathbf{x}_i,\qquad
\pi_k=\frac{N_k}{n}$$

$$\boldsymbol\Sigma_k=\frac{1}{N_k}\sum_i\gamma_{ik}\,(\mathbf{x}_i-\boldsymbol\mu_k)(\mathbf{x}_i-\boldsymbol\mu_k)^\top$$

注意这些就是**加权版的"求均值、求协方差、数个数"**——把 $\gamma_{ik}$ 全设成 0/1 就变回普通公式。
Notice these are just the **weighted versions of "take the mean, the covariance, count"** — set every $\gamma_{ik}$ to 0/1 and they collapse to the ordinary formulas.

**反复 E、M、E、M……** 直到参数不再变化。
**Repeat E, M, E, M…** until the parameters stop changing.

**为什么会收敛**：可以证明每做一次 E+M，数据的（对数）似然都**不会下降**（EM 实际是在优化似然的一个下界，用到 Jensen 不等式）。似然有上界，单调不降 → 一定收敛。但只保证收敛到**局部**最优，所以依赖初始化（sklearn 默认用 K-Means 初始化）。
**Why it converges:** one can prove each E+M round **never decreases** the data (log-)likelihood (EM optimizes a lower bound of it, via Jensen's inequality). The likelihood is bounded above and non-decreasing → it must converge. But only to a **local** optimum, so it depends on initialization (sklearn initializes with K-Means by default).


<a id="4"></a>
## 4. 数据：Old Faithful 间歇泉 / The Old Faithful Geyser Dataset

我们用统计学经典数据集 **Old Faithful**——美国黄石公园"老忠实"间歇泉。
We use the classic **Old Faithful** dataset — the "Old Faithful" geyser in Yellowstone, USA.

每次喷发记录两个数：**喷发持续时长**（分钟）和**距下一次喷发的等待时间**（分钟）。
Each eruption records two numbers: the **eruption duration** (minutes) and the **waiting time** to the next eruption (minutes).

它出名是因为有清晰的**两团结构**：要么"短喷发→短等待"，要么"长喷发→长等待"——是 2 成分 GMM 的教科书例子。下面内联其结构并先把数据画出来认识一下。
It's famous for a clear **two-blob structure**: either "short eruption → short wait" or "long eruption → long wait" — the textbook case for a 2-component GMM. We inline its structure below and plot it first to get familiar.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)

def make_faithful(seed=0):
    rng = np.random.default_rng(seed)
    # 两个真实模式 / two true modes:
    #   短喷发(~2min)+短等待(~54min)，长喷发(~4.3min)+长等待(~80min)
    a = rng.multivariate_normal([2.0, 54], [[0.07, 0.5],[0.5, 35]], 97)
    b = rng.multivariate_normal([4.3, 80], [[0.12, 0.8],[0.8, 45]], 175)
    X = np.vstack([a, b]); rng.shuffle(X)
    return pd.DataFrame(X, columns=["eruptions", "waiting"])

faith = make_faithful()
print(f"Old Faithful: {faith.shape} 次喷发 / eruptions, 2 列 / columns")
print(faith.describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(faith["eruptions"], faith["waiting"], s=18, alpha=0.6)
ax.set_xlabel("喷发时长 eruption duration (min)")
ax.set_ylabel("等待时间 waiting time (min)")
ax.set_title("Old Faithful: 明显两团 / two clear blobs → 2-component GMM")
plt.tight_layout(); plt.show()


<a id="5"></a>
## 5. 从零实现 EM ⭐ / Implementing EM From Scratch

下面把第 3 节的 E 步和 M 步**逐行翻译成代码**。读代码时对照公式：`resp` 就是 $\gamma_{ik}$，`Nk` 就是 $N_k$。
Now we translate the E-step and M-step from Section 3 **line by line into code**. Read alongside the formulas: `resp` is $\gamma_{ik}$, `Nk` is $N_k$.


In [ ]:
X = faith.values

def gmm_em(X, K, n_iter=100, seed=0):
    rng = np.random.default_rng(seed); n, d = X.shape
    # 初始化 / init: 随机选 K 个点当均值, 协方差用全局协方差, 权重均等
    mu = X[rng.choice(n, K, replace=False)]
    Sigma = np.array([np.cov(X.T) for _ in range(K)])
    pi = np.ones(K) / K
    ll_old = -np.inf
    for it in range(n_iter):
        # ---- E 步 / E-step: 算责任 γ_ik (每个点对每个成分的后验概率) ----
        resp = np.array([pi[k] * multivariate_normal(mu[k], Sigma[k]).pdf(X)
                         for k in range(K)]).T            # 分子: 先验×似然
        ll = np.log(resp.sum(1)).sum()                    # 顺便记录对数似然(看收敛)
        resp /= resp.sum(1, keepdims=True)                # 归一化 → Σ_k γ_ik = 1
        # ---- M 步 / M-step: 用责任当权重重估 π, μ, Σ ----
        Nk = resp.sum(0)                                  # 每个成分的"软计数" N_k
        mu = (resp.T @ X) / Nk[:, None]                   # 加权均值
        Sigma = np.array([(resp[:, k, None] * (X - mu[k])).T @ (X - mu[k]) / Nk[k]
                          + 1e-6 * np.eye(d)              # +εI 防协方差奇异
                          for k in range(K)])
        pi = Nk / n                                       # 更新混合权重
        # 收敛判据: 对数似然几乎不再增加就停 / stop when log-likelihood plateaus
        if abs(ll - ll_old) < 1e-4:
            break
        ll_old = ll
    return mu, Sigma, pi, resp, ll

mu, Sigma, pi, resp, ll = gmm_em(X, K=2)
print(f"从零 EM 收敛 / converged. 对数似然 log-likelihood = {ll:.1f}")
print(f"混合权重 mixing weights π = {pi.round(3)}  (应≈ 97/272 和 175/272)")
print(f"两个成分的均值 component means μ:\n{mu.round(1)}")

# 对照 sklearn / compare with sklearn
from sklearn.mixture import GaussianMixture
gm = GaussianMixture(n_components=2, n_init=5, random_state=0).fit(X)
print(f"\nsklearn 均值 means:\n{gm.means_.round(1)}  (与从零一致, 顺序可能不同)")


<a id="6"></a>
## 6. 软分配 + 椭圆簇 / Soft Assignment & Elliptical Clusters

把责任 $\gamma_{i1}$ 当颜色画出来：颜色连续变化，交界处是中间色——这就是**软分配**。再画出两个高斯的等高线，能看到它们是**有方向的椭圆**（K-Means 只能画圆）。
We color points by their responsibility $\gamma_{i1}$: the color varies continuously, with border points in between — that is **soft assignment**. We also draw the two Gaussian contours, which are **tilted ellipses** (K-Means could only draw circles).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左: 软分配 (颜色=属于成分1的概率) / left: soft assignment
axes[0].scatter(X[:, 0], X[:, 1], c=resp[:, 0], cmap="coolwarm", s=22)
axes[0].set_title("软分配 soft assignment\n颜色=属于成分1的概率(边界~0.5) / color = P(component 1)")
axes[0].set_xlabel("eruptions"); axes[0].set_ylabel("waiting")

# 右: 高斯椭圆等高线 / right: Gaussian ellipse contours
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 200),
                     np.linspace(X[:,1].min()-5, X[:,1].max()+5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
dens = sum(pi[k] * multivariate_normal(mu[k], Sigma[k]).pdf(grid) for k in range(2)).reshape(xx.shape)
axes[1].scatter(X[:, 0], X[:, 1], s=14, alpha=0.4)
axes[1].contour(xx, yy, dens, levels=8, cmap="viridis")
axes[1].scatter(mu[:, 0], mu[:, 1], c="red", marker="X", s=150)
axes[1].set_title("拟合出的高斯椭圆(有方向) / fitted tilted Gaussians")
axes[1].set_xlabel("eruptions"); axes[1].set_ylabel("waiting")
plt.tight_layout(); plt.show()
print("软分配给出隶属概率(可表达不确定性); 椭圆协方差能拟合有方向/不同形状的簇")
print("Soft assignment gives membership probabilities; covariance fits tilted/shaped clusters.")


<a id="7"></a>
## 7. GMM 与 K-Means 的关系 ⭐ / GMM vs K-Means

这是面试高频题。**K-Means 其实是 GMM 的一个特例**：
A frequent interview question. **K-Means is actually a special case of GMM:**

- 把每个协方差**固定成 $\sigma^2\mathbf{I}$**（球形、各方向等方差），
  fix every covariance to $\sigma^2\mathbf{I}$ (spherical, equal variance), and
- 让 $\sigma\to 0$，则责任 $\gamma_{ik}$ 退化成 **0/1 的硬分配**（最近的成分拿走全部责任），M 步的加权均值就变成"簇内点求平均"——正是 K-Means。
  let $\sigma\to 0$; then $\gamma_{ik}$ collapses to a **0/1 hard assignment** (the nearest component takes all responsibility), and the M-step weighted mean becomes "average of the points in the cluster" — exactly K-Means.

反过来说，**GMM 就是"软化 + 允许椭圆"的 K-Means**。
Conversely, **GMM is K-Means made soft and allowed to use ellipses.**

实务选择：簇是圆形、大小相近、只要快 → K-Means；簇是椭圆、有重叠、需要概率 → GMM。
In practice: round, equal-size clusters and you just want speed → K-Means; elliptical, overlapping clusters or you need probabilities → GMM.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# 造一组拉长(椭圆)且有重叠的簇 / make elongated, overlapping clusters
rng = np.random.default_rng(3)
T = np.array([[2.5, 1.2], [0, 0.5]])      # 线性变换把圆形拉成斜椭圆
c1 = rng.normal(0, 1, (250, 2)) @ T + [0, 0]
c2 = rng.normal(0, 1, (250, 2)) @ T + [3, 3]
Xe = np.vstack([c1, c2])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(Xe[:,0], Xe[:,1], c=KMeans(2, n_init=10, random_state=0).fit_predict(Xe),
                cmap="coolwarm", s=12)
axes[0].set_title("K-Means: 球形假设 → 在斜椭圆上切错 / spherical → cuts wrong")
axes[1].scatter(Xe[:,0], Xe[:,1], c=GaussianMixture(2, n_init=5, random_state=0).fit_predict(Xe),
                cmap="coolwarm", s=12)
axes[1].set_title("GMM: 椭圆协方差 → 正确分开 / ellipses → correct")
plt.tight_layout(); plt.show()
print("拉长/有方向的簇: K-Means 切错, GMM 正确 → GMM = 软分配+椭圆的 K-Means")
print("Elongated clusters: K-Means fails, GMM succeeds → GMM = soft, elliptical K-Means.")


<a id="8"></a>
## 8. 用 BIC 选簇数 + covariance_type / Choosing K with BIC & covariance_type

K-Means 用肘部/轮廓选 K（6.1）。GMM 是**概率模型**，可以用**信息准则 BIC/AIC**——它们是"对数似然减去复杂度惩罚"，**越低越好**，比聚类肘部更有理论依据。
K-Means picks K with elbow/silhouette (6.1). GMM, being a **probabilistic model**, can use **information criteria BIC/AIC** — "log-likelihood minus a complexity penalty", **lower is better** — more principled than the clustering elbow.

`covariance_type` 控制协方差矩阵的形状（用偏差换方差）：
`covariance_type` controls the shape of the covariance matrix (a bias–variance knob):

- `full`：每个簇任意椭圆（最灵活，参数最多）/ each cluster an arbitrary ellipse (most flexible)
- `diag`：轴对齐椭圆 / axis-aligned ellipse
- `tied`：所有簇共享一个协方差 / all clusters share one covariance
- `spherical`：圆形（最接近 K-Means）/ spherical (closest to K-Means)


In [ ]:
Ks = range(1, 8)
bics = [GaussianMixture(k, n_init=3, random_state=0).fit(X).bic(X) for k in Ks]
aics = [GaussianMixture(k, n_init=3, random_state=0).fit(X).aic(X) for k in Ks]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(Ks), bics, "o-", label="BIC")
ax.plot(list(Ks), aics, "s--", label="AIC")
best = list(Ks)[int(np.argmin(bics))]
ax.axvline(best, color="r", ls=":", label=f"BIC 最优 / best K={best}")
ax.set_xlabel("成分数 number of components K")
ax.set_ylabel("信息准则(越低越好) / info criterion (lower=better)")
ax.legend(); ax.set_title("用 BIC/AIC 选簇数: Old Faithful 选出 2(符合两团) / picks 2")
plt.tight_layout(); plt.show()
print(f"BIC 最优 K = {best} (数据本就 2 个模式 / data really has 2 modes)")
print("covariance_type: full(任意椭圆)/diag(轴对齐)/tied(共享)/spherical(球形≈KMeans)")


<a id="9"></a>
## 9. 小结 / Summary

```
生成故事 / generative story: 先按 π 选一个高斯, 再从该高斯采样 → p(x)=Σ π_k N(x;μ_k,Σ_k)
隐变量 / latent: 不知道每点来自哪个成分 → 鸡生蛋 → EM
EM:
  E 步: 责任 γ_ik = 后验 P(成分k|x_i) = π_k N_k / Σ_j π_j N_j   (软分配)
  M 步: 用 γ 当权重重估 π, μ, Σ (加权均值/协方差)
  反复 E/M, 对数似然单调不降 → 收敛到局部最优(依赖初始化)
软分配(隶属概率) + 椭圆协方差: K-Means 做不到
K-Means = GMM 的 球形等方差 + σ→0(硬分配) 极限
选 K 用 BIC/AIC(越低越好); covariance_type 控椭圆形状
```

### 💡 面试速查 / Interview cheat-sheet
1. **E 步**算责任(软分配, 一个后验概率)，**M 步**用责任加权重估参数。
   E-step computes responsibilities (soft assignment, a posterior); M-step re-estimates parameters weighted by them.
2. **GMM = 软化 + 椭圆的 K-Means**；K-Means 是其球形 + 硬分配极限。
   GMM = soft, elliptical K-Means; K-Means is its spherical, hard-assignment limit.
3. **EM 单调不降对数似然**，收敛到局部最优，依赖初始化(常用 K-Means 初始)。
   EM monotonically increases log-likelihood to a local optimum; depends on init (usually K-Means).
4. **BIC/AIC 选簇数**(有理论依据)；注意协方差要加 εI 防奇异。
   Use BIC/AIC to choose K; add εI to covariance to avoid singularity.

### 下一节 / Next
**6.7 谱聚类**——把数据建成图，用图拉普拉斯的特征向量处理非凸簇。
**6.7 Spectral Clustering** — build a graph and use the Laplacian's eigenvectors for non-convex clusters.
